# Save Model

In [1]:
data = [
  "She composes songs and practices piano daily.",
  "He reads books and explores the nearby caves.",
  "He reads novel and climbs mountains every weekend.",
  "She composes songs and writes novels.",
  "He reads newspaper and solves complex puzzles.",
  "She composes music and organizes exhibitions regularly.",
  "He reads books and builds small wooden models.",
  "He reads books and participates in local science fairs.",
  "She composes songs and curates art projects.",
  "She composes tunes and designs jewelry for her friends.",
  "He reads everyday and documents wildlife photography trips.",
  "She composes harmonies and experiments with digital music.",
  "He reads novel and trains for local marathons.",
  "She composes soundtracks and collaborates with creative filmmakers.",
  "He reads newspaper and studies navigation using maps and stars.",
  "She composes rhythms and teaches music."
]

In [2]:
VOCAB_SIZE = 70
CONTEXT_LEN = 6
EMB_DIM = 24
BATCH_SIZE = 4
EPOCHS = 100

In [3]:
import os
import re
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

In [4]:
save_dir = "models"

In [5]:
_ = torch.manual_seed(123)

In [6]:
def separate_dots(s):
    s = re.sub(r"\.", " . ", s)
    return s.strip()

text_data = [separate_dots(x) for x in data]

In [7]:
example_data = text_data[0]
print(example_data)

She composes songs and practices piano daily .


In [8]:
vocab = set()

for text in text_data:
    for word in text.split():
        vocab.add(word)

vocab = sorted(vocab)

print(len(vocab))

70


In [9]:
file_name = "vocab.json"
os.makedirs(save_dir, exist_ok=True)
with open(os.path.join(save_dir, file_name), "w") as f:
    json.dump(vocab, f)

In [10]:
# Display token:word mappings
for i, word in enumerate(vocab):
    print(f"{i}: {word}")

0: .
1: He
2: She
3: and
4: art
5: books
6: builds
7: caves
8: climbs
9: collaborates
10: complex
11: composes
12: creative
13: curates
14: daily
15: designs
16: digital
17: documents
18: every
19: everyday
20: exhibitions
21: experiments
22: explores
23: fairs
24: filmmakers
25: for
26: friends
27: harmonies
28: her
29: in
30: jewelry
31: local
32: maps
33: marathons
34: models
35: mountains
36: music
37: navigation
38: nearby
39: newspaper
40: novel
41: novels
42: organizes
43: participates
44: photography
45: piano
46: practices
47: projects
48: puzzles
49: reads
50: regularly
51: rhythms
52: science
53: small
54: solves
55: songs
56: soundtracks
57: stars
58: studies
59: teaches
60: the
61: trains
62: trips
63: tunes
64: using
65: weekend
66: wildlife
67: with
68: wooden
69: writes


In [11]:
# build word -> index mapping
word_to_id = {word: idx for idx, word in enumerate(vocab)}

# build id -> word mapping
id_to_word = {idx: word for idx, word in enumerate(vocab)}

In [12]:
def text_to_token_ids(text, word_to_id):
    tokens = text.split()
    return [word_to_id[t] for t in tokens]

def token_ids_to_text(token_ids, id_to_word):
    words = [id_to_word[id] for id in token_ids]
    return " ".join(words)

In [13]:
print(example_data)

She composes songs and practices piano daily .


In [14]:
text_to_token_ids(example_data, word_to_id)

[2, 11, 55, 3, 46, 45, 14, 0]

In [15]:
text_to_token_ids(example_data, word_to_id)[:CONTEXT_LEN + 1]

[2, 11, 55, 3, 46, 45, 14]

In [16]:
text_to_token_ids(example_data, word_to_id)[:CONTEXT_LEN]

[2, 11, 55, 3, 46, 45]

In [17]:
text_to_token_ids(example_data, word_to_id)[1:CONTEXT_LEN + 1]

[11, 55, 3, 46, 45, 14]

In [18]:
class LLMDataset(Dataset):
    def __init__(self, texts, word_to_id, max_len):
        self.texts = texts
        self.word_to_id = word_to_id
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = torch.tensor(text_to_token_ids(self.texts[idx], self.word_to_id)[:self.max_len + 1])
        x = tokens[:-1]
        y = tokens[1:]
        return x, y

# create dataset + dataloader
train_dataset = LLMDataset(
    text_data,
    word_to_id,
    CONTEXT_LEN
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(len(train_dataloader))

4


In [19]:
example_input_output = next(iter(train_dataloader))

In [20]:
print(example_input_output)

[tensor([[ 2, 11, 55,  3, 46, 45],
        [ 1, 49,  5,  3, 22, 60],
        [ 1, 49, 40,  3,  8, 35],
        [ 2, 11, 55,  3, 69, 41]]), tensor([[11, 55,  3, 46, 45, 14],
        [49,  5,  3, 22, 60, 38],
        [49, 40,  3,  8, 35, 18],
        [11, 55,  3, 69, 41,  0]])]


In [21]:
# Example Input
example_input_output[0][0]

tensor([ 2, 11, 55,  3, 46, 45])

In [22]:
# Example Output (shifted by one token)
example_input_output[1][0]

tensor([11, 55,  3, 46, 45, 14])

In [23]:
class Attention(nn.Module):
    def __init__(self, d_in, d_out, context_length):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.register_buffer("mask",torch.triu(torch.ones(context_length, context_length),diagonal=1))

    def forward(self, x, return_weights=False):
        _, num_tokens, _ = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        mask_bool = self.mask[:num_tokens, :num_tokens].bool()

        attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / (self.d_out ** 0.5),dim=-1)

        context_vec = attn_weights @ values

        if return_weights:
            return context_vec, attn_weights

        return context_vec

In [24]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.att = Attention(
            d_in=EMB_DIM,
            d_out=EMB_DIM,
            context_length=CONTEXT_LEN)

    def forward(self, x):
        shortcut = x
        x = self.att(x)
        x = x + shortcut
        return x

In [25]:
class GPTModel(nn.Module):
    def __init__(self):

        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, EMB_DIM)
        self.pos_emb = nn.Embedding(CONTEXT_LEN, EMB_DIM)
        self.trf_block1 = TransformerBlock()
        self.trf_block2 = TransformerBlock()
        self.trf_block3 = TransformerBlock()
        self.trf_block4 = TransformerBlock()
        self.trf_block5 = TransformerBlock()
        self.trf_block6 = TransformerBlock()
        self.output_layer = nn.Linear(EMB_DIM, VOCAB_SIZE, bias=False)

    def forward(self, in_idx):
        _, seq_len = in_idx.shape

        tok_embeds = self.tok_emb(in_idx)

        pos_embeds = self.pos_emb(torch.arange(seq_len))

        x = tok_embeds + pos_embeds

        x = self.trf_block1(x)
        x = self.trf_block2(x)
        x = self.trf_block3(x)
        x = self.trf_block4(x)
        x = self.trf_block5(x)
        x = self.trf_block6(x)

        logits = self.output_layer(x)
        return logits

In [26]:
model = GPTModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.1)
loss_fn = nn.CrossEntropyLoss()

In [27]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {trainable_params:,}")

Total trainable parameters: 13,872


In [28]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 13,872


In [29]:
param_count = defaultdict(int)

for name, param in model.named_parameters():
    top_level = name.split('.')[0]
    param_count[top_level] += param.numel()

for k, v in param_count.items():
    print(f"{k:20s}: {v:,}")

tok_emb             : 1,680
pos_emb             : 144
trf_block1          : 1,728
trf_block2          : 1,728
trf_block3          : 1,728
trf_block4          : 1,728
trf_block5          : 1,728
trf_block6          : 1,728
output_layer        : 1,680


In [30]:
def print_module_params(module, indent=0):
    for name, child in module.named_children():
        print(" " * indent + f"[{name}]")
        for p_name, p in child.named_parameters(recurse=False):
            print(" " * (indent + 2) + f"{p_name}: {p.numel():,}")
        print_module_params(child, indent + 4)

In [31]:
print_module_params(model.trf_block1)

[att]
    [W_query]
      weight: 576
    [W_key]
      weight: 576
    [W_value]
      weight: 576


In [32]:
print_module_params(model.trf_block6)

[att]
    [W_query]
      weight: 576
    [W_key]
      weight: 576
    [W_value]
      weight: 576


In [33]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch, (X, y) in enumerate(dataloader):

        logits = model(X)
        loss = loss_fn(logits.flatten(0, 1), y.flatten())

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        print(f"batch: {batch + 1} loss: {loss:>7f}")

In [34]:
for epoch in range(EPOCHS):
    print("-------------------------------")
    print(f"Epoch: {epoch+1}")
    train(train_dataloader, model, loss_fn, optimizer)
    print("-------------------------------")
print("Done!")

-------------------------------
Epoch: 1
batch: 1 loss: 5.638308
batch: 2 loss: 5.477327
batch: 3 loss: 4.893378
batch: 4 loss: 5.438319
-------------------------------
-------------------------------
Epoch: 2
batch: 1 loss: 5.296641
batch: 2 loss: 5.119512
batch: 3 loss: 4.684375
batch: 4 loss: 5.185306
-------------------------------
-------------------------------
Epoch: 3
batch: 1 loss: 5.034166
batch: 2 loss: 4.824012
batch: 3 loss: 4.505208
batch: 4 loss: 4.962108
-------------------------------
-------------------------------
Epoch: 4
batch: 1 loss: 4.809972
batch: 2 loss: 4.569655
batch: 3 loss: 4.350173
batch: 4 loss: 4.765199
-------------------------------
-------------------------------
Epoch: 5
batch: 1 loss: 4.614145
batch: 2 loss: 4.346789
batch: 3 loss: 4.213614
batch: 4 loss: 4.589357
-------------------------------
-------------------------------
Epoch: 6
batch: 1 loss: 4.439051
batch: 2 loss: 4.148082
batch: 3 loss: 4.090415
batch: 4 loss: 4.429775
------------------

In [35]:
_ = model.eval()

In [36]:
def generate(start_text):
    token_ids = text_to_token_ids(start_text, word_to_id)
    num_new_tokens = CONTEXT_LEN - len(token_ids)
    idx = torch.tensor(token_ids).unsqueeze(0)

    for _ in range(num_new_tokens):
        idx_cond = idx[:, -CONTEXT_LEN:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)

    idx = idx.view(-1).tolist()
    text = token_ids_to_text(idx, id_to_word)
    return text

In [37]:
text = generate("He reads")
print("Generated text:", text)

Generated text: He reads books and explores the


In [38]:
text = generate("She composes")
print("Generated text:", text)

Generated text: She composes songs and practices piano


In [39]:
file_name = "parameters.bin"
os.makedirs(save_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(save_dir, file_name))